In [1]:
!unzip MNIST.zip -d .

Archive:  MNIST.zip
  inflating: ./MNIST/train.csv       


In [ ]:
# MNIST구현
# TensorFlow 구현
# PyTorch 구현



In [3]:
%reset -f

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime #날짜 관련된 모듈, tensorboard에서 이용한 로그를 저장할 폴더명을 만들기 위해서 사용
from sklearn.preprocessing import MinMaxScaler # 데이터핸들링, 전처리, 정규화 등 사용
# 정규화를 편하게 하기위해서 정규화 기법은 크게 2가지만 알아두면 되요
# MinMaxScaler : 값을 0~1 사이로 scaling -> 딥러닝 학습에 유리 데이터의 분포가 깨지는 단점
# StandardScaler : 데이터의 분포를 유지하면서 값을 정규화. 대략 -3~3정도로 분포된다. 
# 일반적인경우(머신러닝같은 경우에는 표준화를 이용해서 정규화하는게 좋아요 분포가 중요하고 데이터의 크기만 줄이는게 목표라 머신러닝은 스탠다드가 더 안정적이고 더 정확하게 학습할 수 있는 정규화다.)
# 딥러닝 같은 경우 학습을 안정적으로 하기위 해 민맥스를 이용해서 정규화하는게 좋다. 
# 특히나 이미지 같은 경우는 민맥스로 정규화를 하는게 좋다.

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.callbacks import TensorBoard

2025-07-02 10:44:42.945195: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-02 10:44:43.072660: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-02 10:44:43.140363: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-02 10:44:43.140784: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-02 10:44:43.219729: I tensorflow/core/platform/cpu_feature_gua

In [5]:
# 데이터 로딩과 전처리
df = pd.read_csv('./MNIST/train.csv')
display(df.head())

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
# 결측치와 이상치는 없어요!
# 데이터를 분리
x_data = df.drop('label', axis=1, inplace=False).values # 2차원 형태의 ndarray를 뽑아내는거다
y_data = df['label'].values # 1차원 ndarray로 

# 학습데이터와 테스트데이터를 분리
x_data_train, x_data_test, y_data_train, y_data_test = train_test_split(x_data,
                                                                       y_data,
                                                                       test_size=0.2,
                                                                       stratify=y_data,
                                                                       random_state=42)
# 정규화
scaler_x = MinMaxScaler()
scaler_x.fit(x_data_train)
x_data_train_norm = scaler_x.transform(x_data_train)
x_data_test_norm = scaler_x.transform(x_data_test)

In [9]:
print(len(x_data_train_norm))

33600


In [11]:
keras_model = Sequential()
keras_model.add(Flatten(input_shape=(784,)))
keras_model.add(Dense(units=128,
                     activation='relu'))
keras_model.add(Dense(units=64,
                     activation='relu'))
keras_model.add(Dense(units=10,
                     activation='softmax'))
keras_model.compile(optimizer=Adam(learning_rate=1e-3),
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])
# callback 설정
es_callback = EarlyStopping(monitor='val_loss',
                           patience=4,
                           restore_best_weights=True,
                           verbose=1)
# 저장하기 위한 ModelCheckpoint
# Tensorboard
log_dir = './logs/'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tb_callback = TensorBoard(log_dir=log_dir,
                         histogram_freq=1)
keras_model.fit(x_data_train_norm,
               y_data_train,
               epochs=100,
               validation_split=0.2,
               callbacks=[es_callback,tb_callback],
               verbose=1,
               batch_size=32) #메모리 사이즈를 생각해서 배치사이즈가 작아지면 속도가 느려진디. 역전파가 많이 사용해지니까 대신 메모리사용량도 작아짐.
# 840이 바로 배치사이즈 결과다. 총 데이터가 33600개이고 여기서 트레인이 80%인 33600*0.8 이고 여기에 배치사이즈 32를 /나눠주면 1에폭당 840이 나온다

Epoch 1/100


I0000 00:00:1751422908.376618    1331 service.cc:145] XLA service 0x7f0ea8005730 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1751422908.376653    1331 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Ti Laptop GPU, Compute Capability 8.6
2025-07-02 11:21:48.397407: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-07-02 11:21:48.496848: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8900


 44/840 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4297 - loss: 1.8646

I0000 00:00:1751422909.772668    1331 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


840/840 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8241 - loss: 0.6047 - val_accuracy: 0.9399 - val_loss: 0.2010
Epoch 2/100
840/840 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9563 - loss: 0.1530 - val_accuracy: 0.9600 - val_loss: 0.1295
Epoch 3/100
840/840 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9711 - loss: 0.0941 - val_accuracy: 0.9632 - val_loss: 0.1247
Epoch 4/100
840/840 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9826 - loss: 0.0614 - val_accuracy: 0.9597 - val_loss: 0.1378
Epoch 5/100
840/840 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9849 - loss: 0.0524 - val_accuracy: 0.9664 - val_loss: 0.1159
Epoch 6/100
840/840 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9899 - loss: 0.0337 - val_accuracy: 0.9643 - val_loss: 0.1203
Epoch 7/100
840/840 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9929 - loss: 0.0274 - val_accuracy: 0.9707 - val_loss: 0.1147
Epoch 8/100
840/840 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9928 - loss: 0.0235 - val_accuracy: 0.9713

In [12]:
# PyTorch 구현
# Tensorflow 같은 경우 GPU설졍(쿠다, 쿠디엔엔 설정)을 했다면 자동으로 쥐피유를 이용해서 학습을 진행 
# 하지만 파이토치는 그렇지 않아요 명시적으로 사용하는 모델과 데이타를 쥐피유에 올려줘야해요
# 파이토치라이트닝은 쥐피유 설정ㅇ이 되어 있담변 쥐피유에서 실행

import torch
import torch.nn as nn # Dense layer를 위해
import torch.optim as optim #Adam Optimizer를 위해
from torch.utils.tensorboard import SummaryWriter #tensorboard를 위해

In [14]:
# GPU를 사용할 수 있는지를 확인하는 처리
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [18]:
# TensorBoard를 위한 로그 파일
log_dir = './logs/'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
writer = SummaryWriter(log_dir=log_dir)

#데이터 변환
# 기존에 가지고 있던 numpy array를 pytorch의 tensor로 변환
x_tensor_train = torch.FloatTensor(x_data_train_norm).to(device)  # 그냥 하면 cpu사용하는데 .to(device)해야 GPU사용됨
x_tensor_test = torch.FloatTensor(x_data_test_norm).to(device)

y_tensor_train = torch.LongTensor(y_data_train).to(device)    #정답데이터가 mnist는 숫자 정수이기 때문에 실수인 FloatTensor가 아닌 정수형인 LongTensor로 설정해야된다 
y_tensor_test = torch.LongTensor(y_data_test).to(device)

# model을 만들어야해요
# tensorflow같은 경우 sequential 같은 것들을 이용해서 생성
# pytorch는 class를 정의해야 해요
# 즉, 모델의 기능을 클래스로 정의한 후 클래스로부터 객체를 생성-> 우리의 모델 객체가 되요
# 클래스는 모델의 기능을 설명하는 거구요. 클래스의 instance(객체)가 실제 동작하는 모델이에요
class MNISTModule(nn.Module):  # nn.Module 클래스를 상속해서 우리 클래스를 정의해요
    def __init__(self): #생성자함수, 초기화 함수, 클래스를 만들면 자동으로 호출하는 함수다 클래스로부터 모델 객체가 생성될 때 자동으로 호출
        super().__init__() # 상위 클래스의 생성자를 호출해서 초기화를 진행
        # 그 다음에는 우리 모델객체가 가지고 있는 Layer를 속성으로 명시 우리 모델안에 내가 사용할 레이어를 변수로 지정한다는 의미
        self.hidden1 = nn.Linear(784, 128)  #Linear가 tensorflow의 Dense layer다  예를 들어서 keras_model.add(Dense(units=128,activation='relu'))
        self.hidden2 = nn.Linear(128, 64)
        self.output = nn.Linear(64,10)
    # 중요한 함수가 하나 나와요 이 함수는 overriding함수에요
    # 순전파 기능을 하는 함수
    def forward(self,x):
        # 위쪽에 명시해 놓은 레이어를 이용해서 순전파를 진행시키면 되요
        # 우리 모델의 예측을 리턴하면 되요
        x = self.hidden1(x)
        x = nn.functional.relu(x)
        x = self.hidden2(x)
        x = nn.functional.relu(x)
        x = self.output(x)
        # softmax activation처리는 여기서 하지 않아요
        return x


# model 객체를 생성
torch_model = MNISTModule().to(device)  # 모델도 쥐피유 메모리에 올려야해요

# 이제 모델을 만들었으니 나머지 설정에 관련된 것들을 만들어야해요
# 모델 설정
# loss 지정
# criterion -> loss 수식을 가지고 있는 변수
criterion = nn.CrossEntropyLoss()  #Tensorflow에서 categorical_crssentropy
optimizer = optim.Adam(torch_model.parameters(), # 모델이 가지고 있는 Weight
                      lr=1e-4)
epochs = 1000

for epoch in range(epochs):
    torch_model.train()  # 모델 학습을 진행하겠다는 의미, 이걸 해줘야 나중에 역전파를 위한 내부 그래프를 생성
    y_pred = torch_model(x_tensor_train)  # 순전파를 진행해서 모델의 예측값을 도출
    loss = criterion(y_pred, y_tensor_train)   #loss를 계산(숫자값을 계산+자료구조도 포함되요)

    #optimizer를 통해서 나중에 수정된 w값을 계산해야해요
    optimizer.zero_grad()  # 가지고 있는 gradient를 0으로 초기화 초기화를 안하면 갱신하면서 누적되서 문제가 생김. 그래서 꼭 초기화를 시켜줘야됨.
    loss.backward()   # 역전파를 진행(gradient만 계산해서 모든 가중치에 대한 gradient를 전파)
    optimizer.step()  # 역전파를 통해 업데이트된 기울기를 이용해서 가중치를 수정!
    writer.add_scalar('Loss/train', loss.item(), epoch) # log 파일에 기록
    if epoch % 1000 == 0:
        print(f'Epochs : {epoch}/{epochs}')
        

    
    
    

        
        
    


Epochs : 0/1000


In [20]:
# 학습이 다 끝나면 평가를 진행
torch_model.eval()  # 모델 안에 있는 Dropout이나 BatchNormalization을 적용하지 않아요
with torch.no_grad():  # 속도를 높이기 위해서
    torch_y_pred = torch_model(x_tensor_test)
    # 확률값으로 나오니까 이 중 가장 높은 확률을 가지는 인덱스가 숫자 이미지에요
    torch_y_pred_class = torch.argmax(torch_y_pred, dim=1).cpu().numpy()
    result_accuracy = accuracy_score(y_tensor_test.cpu(), torch_y_pred_class)
    print(result_accuracy)

0.9298809523809524
